In [14]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.preprocessing import StandardScaler

In [15]:
path = r"C:\Users\Noble Adike\Desktop\Coding\KPMG\kpmg-1a\data\harvard\Year 1 (Jun 2022 - May 2023)\Local_weather_hourly\48190936_Weather.csv"
df = pd.read_csv(path, encoding="cp1252")

In [16]:
df.head()

,Time,"Air temperature (gund, ¡É)","Relative humidity (gund, %)","Wind speed (gund, m/s)","Weather data, Wind speed (m/s)","Wind direction (gund, ¡Æ)","Weather data, Wind direction","Weather data, Rain","Solar radiation (gund, W/m^2)"
0,2022-06-01 00:00:00-04:00,12.576667,88.250000,0.208333,0.033333,98.583333,129.857023,0,1.0
1,2022-06-01 01:00:00-04:00,12.975833,89.908333,0.000000,0.033333,111.083333,129.857023,0,1.0
2,2022-06-01 02:00:00-04:00,13.330833,90.491667,0.000000,0.033333,98.333333,129.857023,0,1.0
3,2022-06-01 03:00:00-04:00,13.648333,87.258333,0.250000,0.050000,84.166667,109.500000,0,1.0
4,2022-06-01 04:00:00-04:00,13.836667,80.758333,0.166667,0.050000,85.833333,119.250157,0,1.0


In [17]:
df.describe()

,"Air temperature (gund, ¡É)","Relative humidity (gund, %)","Wind speed (gund, m/s)","Weather data, Wind speed (m/s)","Wind direction (gund, ¡Æ)","Weather data, Wind direction","Weather data, Rain","Solar radiation (gund, W/m^2)"
count,8760.000000,8760.000000,8760.000000,8760.000000,8760.000000,8760.000000,8760.000000,8760.000000
mean,13.362969,66.762621,1.446469,1.357132,210.244752,206.913554,0.908904,160.201201
std,10.240155,22.106421,1.227710,1.111753,86.092070,106.842196,0.287762,245.939474
min,-24.125000,16.516667,0.000000,0.000000,1.916667,0.177842,0.000000,1.000000
25%,5.261875,48.504167,0.458333,0.355000,146.812500,110.581185,1.000000,1.000000
50%,12.768333,67.516667,1.166667,1.171944,229.416667,263.388614,1.000000,6.083333
75%,20.754167,85.783333,2.166667,2.101042,277.354167,291.194444,1.000000,255.000000
max,42.846667,100.000000,7.491667,6.692778,355.000000,359.634188,1.000000,1109.916667


In [18]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8760 entries, 0 to 8759
Data columns (total 9 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   Time                            8760 non-null   object 
 1   Air temperature (gund, ¡É)      8760 non-null   float64
 2   Relative humidity (gund, %)     8760 non-null   float64
 3   Wind speed (gund, m/s)          8760 non-null   float64
 4   Weather data, Wind speed (m/s)  8760 non-null   float64
 5   Wind direction (gund, ¡Æ)       8760 non-null   float64
 6   Weather data, Wind direction    8760 non-null   float64
 7   Weather data, Rain              8760 non-null   int64  
 8   Solar radiation (gund, W/m^2)   8760 non-null   float64
dtypes: float64(7), int64(1), object(1)
memory usage: 616.1+ KB


In [19]:
df['Time'] = pd.to_datetime(df['Time'])
df = df.set_index('Time')

C:\Users\Noble Adike\AppData\Local\Temp\ipykernel_17528\1925714391.py:1: FutureWarning: In a future version of pandas, parsing datetimes with mixed time zones will raise an error unless `utc=True`. Please specify `utc=True` to opt in to the new behaviour and silence this warning. To create a `Series` with mixed offsets and `object` dtype, please use `apply` and `datetime.datetime.strptime`
  df['Time'] = pd.to_datetime(df['Time'])


In [20]:
df = df.rename(columns={
    'Air temperature (gund, ¡É)': 'air_temp_c',
    'Relative humidity (gund, %)': 'rel_humidity_pct',
    'Wind speed (gund, m/s)': 'wind_speed_sensor',
    'Weather data, Wind speed (m/s)': 'wind_speed_weather',
    'Wind direction (gund, ¡Æ)': 'wind_dir_sensor_deg',
    'Weather data, Wind direction': 'wind_dir_weather_deg',
    'Weather data, Rain': 'rain_flag',
    'Solar radiation (gund, W/m^2)': 'solar_ghi_wm2'
})

In [21]:
df['rel_humidity_pct'] = df['rel_humidity_pct'].clip(0, 100)
for c in ['wind_speed_sensor','wind_speed_weather','solar_ghi_wm2']:
    df[c] = df[c].clip(lower=0)

In [22]:
df.isna().sum()

air_temp_c              0
rel_humidity_pct        0
wind_speed_sensor       0
wind_speed_weather      0
wind_dir_sensor_deg     0
wind_dir_weather_deg    0
rain_flag               0
solar_ghi_wm2           0
dtype: int64

In [23]:
good= ['air_temp_c','rel_humidity_pct','wind_speed_sensor','wind_speed_weather','wind_dir_sensor_deg','wind_dir_weather_deg','solar_ghi_wm2']
def treat_outliers_iqr(df):
    df_clean = df.copy()
    for col in df_clean.columns:
        Q1 = df_clean[col].quantile(0.25)
        Q3 = df_clean[col].quantile(0.75)
        IQR = Q3 - Q1


        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        df_clean[col] = df_clean[col].clip(lower=lower_bound, upper=upper_bound)
    return df_clean
df[good] = treat_outliers_iqr(df[good])

In [24]:
df.describe()

,air_temp_c,rel_humidity_pct,wind_speed_sensor,wind_speed_weather,wind_dir_sensor_deg,wind_dir_weather_deg,rain_flag,solar_ghi_wm2
count,8760.000000,8760.000000,8760.000000,8760.000000,8760.000000,8760.000000,8760.000000,8760.000000
mean,13.369461,66.762621,1.434032,1.354765,210.244752,206.913554,0.908904,148.317658
std,10.218599,22.106421,1.187982,1.103479,86.092070,106.842196,0.287762,215.295665
min,-17.976563,16.516667,0.000000,0.000000,1.916667,0.177842,0.000000,1.000000
25%,5.261875,48.504167,0.458333,0.355000,146.812500,110.581185,1.000000,1.000000
50%,12.768333,67.516667,1.166667,1.171944,229.416667,263.388614,1.000000,6.083333
75%,20.754167,85.783333,2.166667,2.101042,277.354167,291.194444,1.000000,255.000000
max,42.846667,100.000000,4.729167,4.720104,355.000000,359.634188,1.000000,636.000000


In [25]:
list(df.columns)

['air_temp_c',
 'rel_humidity_pct',
 'wind_speed_sensor',
 'wind_speed_weather',
 'wind_dir_sensor_deg',
 'wind_dir_weather_deg',
 'rain_flag',
 'solar_ghi_wm2']